# Predicting Next-Day S&P 500 Excess Returns  
## A GitHub-Friendly Project Notebook

**Course:** STAT 306  
**Group members:** Winnie Chen, Jeffrey Deng, Wendy Xiao, Jingyi Yang

---

## What this notebook is
This notebook is a polished, repository-friendly version of the project workflow. It combines:

- clear written explanation,
- the full R analysis pipeline,
- model interpretation,
- diagnostic checks,
- robustness checks, and
- reproducibility notes.

It is designed so that a reader on GitHub can understand both **what was done** and **why it was done** without needing to jump back and forth between code and a separate report.

## Main research question
Do simple recent market state variables help explain **next-day S&P 500 excess returns**?

More specifically, the project studies whether the following quantities observed at the end of day `t` are associated with the excess return on day `t + 1`:

1. `ExcessReturn_t`: the excess return on the current day,
2. `RollingMean_5`: the 5-day rolling mean of excess returns,
3. `RollingVolatility_5`: the 5-day rolling standard deviation of excess returns.

## Data sources
The analysis combines two public sources:

- **Yahoo Finance** for daily S&P 500 index data (`^GSPC`)
- **FRED** for the 3-Month Treasury Bill Secondary Market Rate (`DTB3`)

The response variable is **next-day excess return**, and every predictor is built only from information available by the close of day `t`. This avoids look-ahead bias.

## Main takeaway
The project finds **very limited evidence of predictable next-day linear structure**. The only recurring pattern is a weak negative coefficient on the 5-day rolling mean, suggesting mild short-run reversal. But the effect is fragile, the fit is extremely low, and the signal weakens under nearby specification changes.

---


## 0. Package setup

This section loads all required R packages. The code below is written to be relatively reproducible: if a package is missing, it will be installed and then loaded.

The workflow uses:

- `quantmod` for market and FRED data,
- `dplyr` and `tibble` for data wrangling,
- `zoo` for rolling-window calculations,
- `ggplot2` for plots,
- `car` for VIF diagnostics,
- `broom` for tidy regression outputs,
- `readr` for file import/export.


In [ ]:
required_packages <- c(
  "quantmod",
  "dplyr",
  "zoo",
  "ggplot2",
  "car",
  "broom",
  "tibble",
  "readr"
)

for (pkg in required_packages) {
  if (!require(pkg, character.only = TRUE, quietly = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}


## 1. File settings

The notebook first checks whether a locally cleaned file already exists. If it does, the analysis uses that file directly. Otherwise, the notebook rebuilds the cleaned dataset from raw downloaded sources.

This makes the notebook useful in two ways:

- **fast reruns** when the cleaned dataset is already available,
- **full reproducibility** when the cleaned file is absent.


In [ ]:
clean_data_file <- "SP500_data_cleaned.csv"


## 2. Data loading and construction

This is the core data-engineering section.

### What happens here
If a cleaned file is already available, we read it in. Otherwise, the notebook:

1. downloads S&P 500 daily data from Yahoo Finance,
2. downloads the 3-Month Treasury Bill rate from FRED,
3. converts daily index prices into daily returns,
4. converts the annualized Treasury Bill series into an approximate daily risk-free rate,
5. computes excess returns,
6. creates lagged and rolling-window predictors,
7. aligns the predictors at time `t` with the response at time `t + 1`,
8. drops incomplete rows after all variables are created.

### Why the timing matters
The response is `NextDayExcessReturn`, while all predictors are known by the end of day `t`. This matters because it preserves the forecasting interpretation and avoids using future information.

### Variables created
- `Returns`: simple daily S&P 500 return
- `RiskFreeRate`: approximate daily risk-free rate
- `ExcessReturn`: `Returns - RiskFreeRate`
- `ExcessReturn_t`: current-day excess return used as predictor
- `RollingMean_5`: 5-day average excess return
- `RollingVolatility_5`: 5-day standard deviation of excess returns
- `NextDayExcessReturn`: next-day excess return, used as the response


In [ ]:
if (file.exists(clean_data_file)) {
  message("Loading local cleaned data file: ", clean_data_file)
  data_final <- readr::read_csv(clean_data_file, show_col_types = FALSE)
  data_final$Date <- as.Date(data_final$Date)

} else {
  message("Local cleaned data file not found. Downloading raw data and constructing dataset...")

  # ----------------------------
  # 2A. Download S&P 500 data
  # ----------------------------
  # Use Yahoo Finance ticker ^GSPC and closing price series.
  quantmod::getSymbols("^GSPC", src = "yahoo", from = "2022-01-01", auto.assign = TRUE)

  sp500_prices <- data.frame(
    Date = as.Date(index(GSPC)),
    Price = as.numeric(Cl(GSPC))
  ) %>%
    arrange(Date) %>%
    mutate(
      Returns = (Price / lag(Price)) - 1
    )

  # ----------------------------
  # 2B. Download risk-free rate data
  # ----------------------------
  # FRED DTB3: annualized percentage rate.
  quantmod::getSymbols("DTB3", src = "FRED", auto.assign = TRUE)

  rf_data <- data.frame(
    Date = as.Date(index(DTB3)),
    DTB3 = as.numeric(DTB3)
  )

  # ----------------------------
  # 2C. Merge raw data by date
  # ----------------------------
  merged_data <- sp500_prices %>%
    left_join(rf_data, by = "Date") %>%
    arrange(Date) %>%
    mutate(
      DTB3 = zoo::na.locf(DTB3, na.rm = FALSE),
      RiskFreeRate = (DTB3 / 100) / 252
    )

  # ----------------------------
  # 2D. Construct project variables
  # ----------------------------
  # Each row is trading day t.
  # Explanatory variables use information available by end of day t.
  # Response is next-day excess return on day t+1.
  data_final <- merged_data %>%
    arrange(Date) %>%
    mutate(
      ExcessReturn = Returns - RiskFreeRate,
      ExcessReturn_t = ExcessReturn,
      RollingMean_5 = zoo::rollapply(
        ExcessReturn,
        width = 5,
        FUN = mean,
        align = "right",
        fill = NA,
        na.rm = TRUE
      ),
      RollingVolatility_5 = zoo::rollapply(
        ExcessReturn,
        width = 5,
        FUN = sd,
        align = "right",
        fill = NA,
        na.rm = TRUE
      ),
      NextDayExcessReturn = lead(ExcessReturn, 1)
    ) %>%
    select(
      Date,
      Price,
      Returns,
      DTB3,
      RiskFreeRate,
      ExcessReturn,
      ExcessReturn_t,
      RollingMean_5,
      RollingVolatility_5,
      NextDayExcessReturn
    ) %>%
    filter(
      !is.na(Returns),
      !is.na(RiskFreeRate),
      !is.na(ExcessReturn_t),
      !is.na(RollingMean_5),
      !is.na(RollingVolatility_5),
      !is.na(NextDayExcessReturn)
    )

  # Save cleaned dataset for reproducibility
  readr::write_csv(data_final, clean_data_file)
}

# Basic data check
print(head(data_final))
print(summary(data_final))
cat("Number of observations:", nrow(data_final), "\n")


## 3. Exploratory data analysis

Before fitting any model, we examine the cleaned dataset visually and numerically.

### What we want to learn here
The exploratory analysis checks:

- whether excess returns are centered near zero,
- whether the data look highly noisy,
- whether there is evidence of volatility clustering,
- whether the candidate predictors show any obvious linear pattern with next-day excess returns.

### Interpreting the EDA
In financial data, a low signal-to-noise environment is common. So the purpose of the plots is not to "prove" predictive relationships, but to see whether any pattern looks plausible enough to motivate formal modeling.

Typical expectations:
- the return series should fluctuate around zero,
- the histogram should be sharply centered with heavier tails,
- rolling mean may show a slight slope if short-run reversal or momentum is present,
- rolling volatility often looks noisy and may not show strong direct linear association with next-day returns.


In [ ]:
plot_excess_time <- ggplot(data_final, aes(x = Date, y = ExcessReturn)) +
  geom_line(color = "steelblue") +
  geom_hline(yintercept = 0, color = "red", linetype = "dashed") +
  labs(
    title = "Time Series of S&P 500 Excess Returns",
    x = "Date",
    y = "Excess Return"
  )
print(plot_excess_time)

ggsave("plot_excess_returns_time_series.png", plot_excess_time, width = 8, height = 5, dpi = 300)

plot_nextday_hist <- ggplot(data_final, aes(x = NextDayExcessReturn)) +
  geom_histogram(bins = 40, fill = "skyblue", color = "white") +
  labs(
    title = "Distribution of Next-Day Excess Returns",
    x = "Next-Day Excess Return",
    y = "Count"
  )
print(plot_nextday_hist)

ggsave("plot_nextday_excess_return_hist.png", plot_nextday_hist, width = 8, height = 5, dpi = 300)

plot_vol_hist <- ggplot(data_final, aes(x = RollingVolatility_5)) +
  geom_histogram(bins = 40, fill = "orange", color = "white") +
  labs(
    title = "Distribution of 5-Day Rolling Volatility",
    x = "5-Day Rolling Volatility",
    y = "Count"
  )
print(plot_vol_hist)

ggsave("plot_rolling_volatility_hist.png", plot_vol_hist, width = 8, height = 5, dpi = 300)

plot_scatter_t <- ggplot(data_final, aes(x = ExcessReturn_t, y = NextDayExcessReturn)) +
  geom_point(alpha = 0.3, size = 0.8) +
  geom_smooth(method = "lm", se = TRUE) +
  labs(
    title = "Day t Excess Return vs Next-Day Excess Return",
    x = "Excess Return on Day t",
    y = "Excess Return on Day t+1"
  )
print(plot_scatter_t)

ggsave("plot_excessreturn_t_vs_nextday.png", plot_scatter_t, width = 8, height = 5, dpi = 300)

plot_scatter_mean <- ggplot(data_final, aes(x = RollingMean_5, y = NextDayExcessReturn)) +
  geom_point(alpha = 0.3, size = 0.8) +
  geom_smooth(method = "lm", se = TRUE) +
  labs(
    title = "5-Day Rolling Mean vs Next-Day Excess Return",
    x = "5-Day Rolling Mean of Excess Returns",
    y = "Excess Return on Day t+1"
  )
print(plot_scatter_mean)

ggsave("plot_rollingmean5_vs_nextday.png", plot_scatter_mean, width = 8, height = 5, dpi = 300)

plot_scatter_vol <- ggplot(data_final, aes(x = RollingVolatility_5, y = NextDayExcessReturn)) +
  geom_point(alpha = 0.3, size = 0.8) +
  geom_smooth(method = "lm", se = TRUE) +
  labs(
    title = "5-Day Rolling Volatility vs Next-Day Excess Return",
    x = "5-Day Rolling Volatility of Excess Returns",
    y = "Excess Return on Day t+1"
  )
print(plot_scatter_vol)

ggsave("plot_rollingvol5_vs_nextday.png", plot_scatter_vol, width = 8, height = 5, dpi = 300)


## 4. Regression models

This project compares four linear models.

### Model definitions
- **Model 1:** `NextDayExcessReturn ~ ExcessReturn_t`
- **Model 2:** `NextDayExcessReturn ~ RollingMean_5`
- **Model 3 (main model):** `NextDayExcessReturn ~ RollingMean_5 + RollingVolatility_5`
- **Model 4:** `NextDayExcessReturn ~ ExcessReturn_t + RollingMean_5 + RollingVolatility_5`

### Why this sequence?
The model sequence is intentional.

- Model 1 asks whether the most recent daily movement alone carries information.
- Model 2 isolates the short-horizon average effect.
- Model 3 checks whether recent volatility adds anything once recent direction is already included.
- Model 4 asks whether the single most recent day adds anything beyond the rolling summaries.

This structure lets us separate the contribution of each predictor rather than jumping directly to one larger specification.


In [ ]:
model_1 <- lm(
  NextDayExcessReturn ~ ExcessReturn_t,
  data = data_final
)

model_2 <- lm(
  NextDayExcessReturn ~ RollingMean_5,
  data = data_final
)

model_3 <- lm(
  NextDayExcessReturn ~ RollingMean_5 + RollingVolatility_5,
  data = data_final
)

model_4 <- lm(
  NextDayExcessReturn ~ ExcessReturn_t + RollingMean_5 + RollingVolatility_5,
  data = data_final
)

# Main specification and fuller comparison model
main_model <- model_3
comparison_model <- model_4

cat("\n===== Model 1 =====\n")
print(summary(model_1))
cat("\n===== Model 2 =====\n")
print(summary(model_2))
cat("\n===== Model 3 (Main Model) =====\n")
print(summary(model_3))
cat("\n===== Model 4 (Comparison Model) =====\n")
print(summary(model_4))


## 5. Readable regression tables

The code below converts the raw model output into cleaner tables for interpretation and export.

### Why this matters
Default regression summaries are useful, but they are not ideal for reports or GitHub repositories. Tidy coefficient tables make it easier to compare:

- coefficient estimates,
- standard errors,
- test statistics,
- p-values,
- model-level fit summaries.

The exported CSV files are also useful if the repository later includes a `results/` folder.


In [ ]:
coef_table <- bind_rows(
  broom::tidy(model_1) %>% mutate(Model = "Model 1: Day t excess return only"),
  broom::tidy(model_2) %>% mutate(Model = "Model 2: Rolling mean only"),
  broom::tidy(model_3) %>% mutate(Model = "Model 3: Rolling mean + volatility (main)"),
  broom::tidy(model_4) %>% mutate(Model = "Model 4: Full comparison model")
) %>%
  select(Model, term, estimate, std.error, statistic, p.value)

print(coef_table)
readr::write_csv(coef_table, "regression_coefficients_table.csv")

get_fit_stats <- function(model, label) {
  s <- summary(model)
  tibble(
    Model = label,
    R_squared = s$r.squared,
    Adjusted_R_squared = s$adj.r.squared,
    Residual_SE = s$sigma,
    F_statistic = s$fstatistic[1],
    DF1 = s$fstatistic[2],
    DF2 = s$fstatistic[3],
    F_p_value = pf(s$fstatistic[1], s$fstatistic[2], s$fstatistic[3], lower.tail = FALSE)
  )
}

fit_table <- bind_rows(
  get_fit_stats(model_1, "Model 1: Day t excess return only"),
  get_fit_stats(model_2, "Model 2: Rolling mean only"),
  get_fit_stats(model_3, "Model 3: Rolling mean + volatility (main)"),
  get_fit_stats(model_4, "Model 4: Full comparison model")
)

print(fit_table)
readr::write_csv(fit_table, "regression_fit_table.csv")


## 6. Diagnostics for the main model

The main model is a simple linear regression, so we need to check whether its assumptions are approximately reasonable.

### Diagnostics examined
The notebook produces:

- residuals vs fitted,
- normal Q-Q plot,
- scale-location plot,
- residuals vs leverage,
- residuals over time.

### What we are looking for
These plots help us assess:

- possible nonlinearity,
- non-normal tails,
- heteroskedasticity,
- influential points,
- time clustering in variability.

For daily market returns, departures from textbook OLS assumptions are not surprising. In fact, volatility clustering and heavy tails are common, so these plots are especially important for avoiding overconfident interpretation.


In [ ]:
png("main_model_diagnostic_plots.png", width = 1200, height = 1200, res = 150)
par(mfrow = c(2, 2))
plot(main_model)
par(mfrow = c(1, 1))
dev.off()

# Also show in console session
par(mfrow = c(2, 2))
plot(main_model)
par(mfrow = c(1, 1))

data_final$residuals_main <- resid(main_model)

plot_resid_time <- ggplot(data_final, aes(x = Date, y = residuals_main)) +
  geom_line(color = "purple") +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") +
  labs(
    title = "Residuals Over Time for Main Model",
    x = "Date",
    y = "Residual"
  )
print(plot_resid_time)

ggsave("plot_residuals_over_time_main_model.png", plot_resid_time, width = 8, height = 5, dpi = 300)

cooks_main <- cooks.distance(main_model)
top_influential <- tibble(
  Row = seq_along(cooks_main),
  Date = data_final$Date,
  CooksD = cooks_main
) %>%
  arrange(desc(CooksD)) %>%
  slice(1:10)

cat("\nTop 10 influential observations by Cook's distance (main model):\n")
print(top_influential)
readr::write_csv(top_influential, "top_influential_points_main_model.csv")


## 7. Predictor overlap, VIF, and nested model comparison

Because all predictors are derived from recent returns, some overlap is expected.

### Why this section matters
A weak result can come from at least two different sources:

1. there is genuine signal, but predictors are too collinear to estimate it precisely, or  
2. there is simply very little signal in the data.

To separate those possibilities, this section checks:

- the predictor correlation matrix,
- the correlation between `ExcessReturn_t` and `RollingMean_5`,
- variance inflation factors (VIFs),
- nested model ANOVA comparisons.

### Interpretation goal
If VIF values are low and added predictors still fail to improve fit, then the main problem is probably **weak explanatory power**, not multicollinearity.


In [ ]:
predictor_cor <- data_final %>%
  select(ExcessReturn_t, RollingMean_5, RollingVolatility_5) %>%
  cor(use = "complete.obs")

cat("\nPredictor correlation matrix:\n")
print(predictor_cor)
write.csv(predictor_cor, "predictor_correlation_matrix.csv")

cor_excess_rollmean <- cor(
  data_final$ExcessReturn_t,
  data_final$RollingMean_5,
  use = "complete.obs"
)

cat("\nCorrelation between ExcessReturn_t and RollingMean_5:",
    round(cor_excess_rollmean, 4), "\n")

vif_values <- car::vif(comparison_model)
cat("\nVIF values for comparison model:\n")
print(vif_values)
write.csv(as.data.frame(vif_values), "vif_values_comparison_model.csv")

cat("\nNested model comparison: Model 2 vs Model 3\n")
print(anova(model_2, model_3))

cat("\nNested model comparison: Model 3 vs Model 4\n")
print(anova(model_3, model_4))

adjr2_main <- summary(main_model)$adj.r.squared
p_main <- pf(
  summary(main_model)$fstatistic[1],
  summary(main_model)$fstatistic[2],
  summary(main_model)$fstatistic[3],
  lower.tail = FALSE
)

adjr2_comp <- summary(comparison_model)$adj.r.squared
p_comp <- pf(
  summary(comparison_model)$fstatistic[1],
  summary(comparison_model)$fstatistic[2],
  summary(comparison_model)$fstatistic[3],
  lower.tail = FALSE
)

cat("\nMain model adjusted R-squared =", round(adjr2_main, 4), "\n")
cat("Main model overall F-test p-value =", round(p_main, 4), "\n")
cat("Comparison model adjusted R-squared =", round(adjr2_comp, 4), "\n")
cat("Comparison model overall F-test p-value =", round(p_comp, 4), "\n")

if (abs(cor_excess_rollmean) > 0.3) {
  cat("Interpretation: ExcessReturn_t and RollingMean_5 show noticeable conceptual overlap.\n")
}

if (adjr2_main >= adjr2_comp) {
  cat("Interpretation: The simpler main model is at least as competitive as the fuller comparison model.\n")
} else {
  cat("Interpretation: The fuller comparison model does not clearly improve explanatory power.\n")
}


## 8. Rolling-window robustness check

The project's most interesting coefficient is the one on the rolling mean. But a result is more convincing when it remains reasonably stable under nearby modeling choices.

### What this check does
This section refits the main regression idea using different rolling windows:

- 3-day,
- 5-day,
- 10-day.

### Why this matters
If the sign and significance of the rolling-mean effect change a lot when the window changes slightly, then the result should be treated as fragile rather than robust.


In [ ]:
# This checks whether the weak RollingMean signal depends heavily on the 5-day window.

run_window_model <- function(window_size, base_data) {
  temp_data <- base_data %>%
    arrange(Date) %>%
    mutate(
      RollingMean = zoo::rollapply(
        ExcessReturn,
        width = window_size,
        FUN = mean,
        align = "right",
        fill = NA,
        na.rm = TRUE
      ),
      RollingVolatility = zoo::rollapply(
        ExcessReturn,
        width = window_size,
        FUN = sd,
        align = "right",
        fill = NA,
        na.rm = TRUE
      ),
      NextDayExcessReturn = lead(ExcessReturn, 1)
    ) %>%
    filter(
      !is.na(RollingMean),
      !is.na(RollingVolatility),
      !is.na(NextDayExcessReturn)
    )

  fit <- lm(
    NextDayExcessReturn ~ RollingMean + RollingVolatility,
    data = temp_data
  )

  fit_tidy <- broom::tidy(fit)
  fit_summary <- summary(fit)

  tibble(
    Window = window_size,
    RollingMean_Estimate = fit_tidy$estimate[fit_tidy$term == "RollingMean"],
    RollingMean_p_value = fit_tidy$p.value[fit_tidy$term == "RollingMean"],
    RollingVolatility_Estimate = fit_tidy$estimate[fit_tidy$term == "RollingVolatility"],
    RollingVolatility_p_value = fit_tidy$p.value[fit_tidy$term == "RollingVolatility"],
    Adjusted_R_squared = fit_summary$adj.r.squared,
    Overall_F_p_value = pf(
      fit_summary$fstatistic[1],
      fit_summary$fstatistic[2],
      fit_summary$fstatistic[3],
      lower.tail = FALSE
    )
  )
}

# Ensure base data has ExcessReturn for robustness runs
base_for_robustness <- data_final %>%
  select(Date, ExcessReturn)

window_results <- bind_rows(
  run_window_model(3, base_for_robustness),
  run_window_model(5, base_for_robustness),
  run_window_model(10, base_for_robustness)
)

cat("\nRolling-window robustness results:\n")
print(window_results)
readr::write_csv(window_results, "rolling_window_robustness.csv")


## 9. Sensitivity check for influential observations

Borderline significance can sometimes be driven by only a few unusual days.

### What this check does
The notebook identifies the most influential observations and refits the main model after removing the top five.

### Why this matters
If the apparent signal disappears after removing a handful of influential points, then the original result is not very stable. In that case, the most honest conclusion is that the data provide only weak or fragile evidence.


In [ ]:
# This is still within a simple linear regression workflow.
# We refit the main model after removing the 5 most influential points.

rows_to_drop <- top_influential$Row[1:5]
data_sensitivity <- data_final[-rows_to_drop, ]

main_model_sensitivity <- lm(
  NextDayExcessReturn ~ RollingMean_5 + RollingVolatility_5,
  data = data_sensitivity
)

cat("\n===== Main model after removing top 5 influential points =====\n")
print(summary(main_model_sensitivity))

sensitivity_table <- broom::tidy(main_model_sensitivity)
readr::write_csv(sensitivity_table, "main_model_sensitivity_after_dropping_top5.csv")


## 10. Interpretation and conclusions

### Summary of findings
Across the full workflow, the overall message is cautious:

- `ExcessReturn_t` does **not** provide strong evidence of next-day predictive power.
- `RollingVolatility_5` is generally not statistically important in the linear models used here.
- `RollingMean_5` is the only predictor that shows a repeated negative association with next-day excess returns.

That negative sign is consistent with a **mild short-run reversal** story: after several relatively strong recent days, the next day tends to be slightly weaker on average.

### Why the result should be treated carefully
Even where the rolling-mean coefficient is statistically significant, several facts argue for caution:

- adjusted R-squared values are extremely close to zero,
- the practical effect size is small,
- the signal weakens when the rolling window changes,
- the signal also weakens after influential observations are removed,
- the residual behavior is consistent with heavy tails and changing volatility.

### Bottom line
The data contain at most a **small and unstable linear signal**. The project therefore does **not** support a strong claim that a few recent return summaries can meaningfully explain next-day S&P 500 excess returns.

### Good next steps
A stronger follow-up project could explore:

- out-of-sample testing,
- robust standard errors,
- nonlinear specifications,
- GARCH-type volatility models,
- richer predictor sets such as volume, implied volatility, or macro surprises.


## 11. Session information

This final section records the R session details so that the software environment can be documented for reproducibility.


In [ ]:
cat("\n===== Session Info =====\n")
print(sessionInfo())
